
# SURFMATCH — NYLON_PLANE vs NYLON_SPHERE / SURFMAP
## Corrected quantitative comparison notebook

This notebook compares two 2D representations of the **same physical nylon shell points**:

- **NYLON_PLANE**: orthogonal projection onto the physical nylon face, expressed in Å.
- **NYLON_SPHERE / SURFMAP**: the coordinates produced by SURFMAP.

The main scientific question is:

> **How much geometric and electrostatic spatial information is altered when the approximately planar nylon face is represented through SURFMAP?**

---

## Why this notebook is different from the previous version

The previous version was useful for the first comparison, but it mixed some effects that should be separated.

This corrected version makes four important changes:

### 1. The alignment step is made explicit

Before comparing the shape of two maps, we remove differences that do **not** represent true deformation:

- translation;
- rotation;
- optional reflection;
- one uniform global scale.

This operation is called a **similarity alignment**.

It is equivalent to printing the two maps on transparent sheets and allowing one sheet to be:

- moved;
- rotated;
- flipped if necessary;
- uniformly zoomed.

It is **not** allowed to be bent, stretched differently along different directions, or warped locally.

Therefore, any residual difference after this alignment represents a true difference in shape.

### 2. SURFMAP discretization is measured explicitly

The SURFMAP `coord_list` may already contain quantized/grid coordinates. If many physical points share exactly the same 2D coordinate, then local point-level metrics are affected not only by the spherical mapping but also by map discretization.

This notebook therefore reports:

- number of unique SURFMAP 2D positions;
- fraction of points involved in coordinate collisions;
- maximum number of physical points sharing one SURFMAP position;
- effective physical size of one SURFMAP coordinate step after alignment.

The global deformation is still meaningful, but very fine local metrics are interpreted separately.

### 3. Local deformation is evaluated at several physical length scales

Instead of relying mainly on nearest-neighbour distances or tiny Delaunay triangles, the notebook measures pairwise distance distortion in physical bands such as:

- 0.5–1 Å
- 1–2 Å
- 2–5 Å
- 5–10 Å
- 10–15 Å

This allows us to distinguish loss of sub-pixel detail from deformation of larger spatial patterns that are more relevant to SURFMATCH.

### 4. Charge-pattern preservation is evaluated at several physical resolutions

Duplicate SURFMAP coordinates make direct interpolation of individual points potentially unstable.

The corrected approach first rasterizes both representations on the **same physical grid**, explicitly averaging points that fall in the same cell, and then applies the same Gaussian spatial smoothing to both maps.

Charge similarity is then measured at different physical smoothing scales.

This asks a more useful question:

> **Are electrostatic patterns preserved at 1 Å, 2 Å, 5 Å, etc.?**

---

## Important distinction

If the SURFMAP coordinates available in `coord_list` are already integer/grid coordinates, the notebook measures the **effective error of the SURFMAP representation actually available for matching**:

\[
\text{effective error}
=
\text{projection deformation}
+
\text{coordinate quantization/binning}
\]

This is directly relevant if SURFMATCH would use that SURFMAP map.

It should not, however, be described as the *pure spherical-projection error alone* unless continuous pre-binning SURFMAP coordinates are available.



## 1. Imports

The notebook uses only standard numerical/scientific Python packages.

```bash
pip install numpy pandas scipy plotly
```


In [3]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter

print("Libraries loaded.")


Libraries loaded.



## 2. User settings

Only file paths and explicitly documented analysis scales should normally be changed.

### Why no 25-pixel ROI is used here

A radius of 25 **pixels** in SURFMAP cannot automatically be treated as 25 Å in NYLON_PLANE.

Before defining a ROI correspondence, we first measure the deformation of the common physical points and evaluate how well **central physical regions** can be aligned at different radii in Å.

This provides the quantitative basis for a later ROI-specific notebook.


In [4]:

# ============================================================
# NYLON_PLANE INPUTS
# ============================================================

PLANE_OUTPUT_DIR = Path(
     "/home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection"
)

PLANE_POINTS_CSV = (
    PLANE_OUTPUT_DIR / "/home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_planar_projection_corrected/nylon_planar_projection_points.csv"
)

PLANE_METADATA_JSON = (
    PLANE_OUTPUT_DIR / "/home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_planar_projection_corrected/nylon_planar_projection_metadata.json"
)

# ============================================================
# SURFMAP INPUTS
# ============================================================

SURFMAP_OUTPUT_DIR = Path(
    "/home/chiara/Desktop/COMENZE/Comenze_pipeline/"
    "Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/"
    "output_SURFMAP_NYL_newVer_rotY90resnum_bfactor"
)

# Leave as None for conservative automatic discovery.
SURFMAP_COORD_LIST = None
SURFMAP_SHELL_PDB = None

# ============================================================
# POINT CORRESPONDENCE
# ============================================================

XYZ_MATCH_TOL_A = 0.002

# Used only when a physical point appears several times around
# a SURFMAP seam.
SURFMAP_CENTER_2D = np.array([0.0, 0.0])

# ============================================================
# GEOMETRIC ANALYSIS
# ============================================================

# Physical distance bands used for multiscale pair analysis.
DISTANCE_BANDS_A = [
    (0.0, 0.5),
    (0.5, 1.0),
    (1.0, 2.0),
    (2.0, 5.0),
    (5.0, 10.0),
    (10.0, 15.0),
]

# Number of nearest physical neighbours retained per point.
# A relatively high value gives access to several Å length scales
# without constructing all N(N-1)/2 possible point pairs.
PAIR_K_NEIGHBORS = 500

# Central physical radii evaluated independently.
CENTRAL_RADII_A = [
    5, 10, 15, 20, 25, 30, 35, 40, 45
]

# ============================================================
# CHARGE-PATTERN ANALYSIS
# ============================================================

# Common physical raster grid.
CHARGE_GRID_STEP_A = 0.5

# Same smoothing is applied to plane and SURFMAP fields.
CHARGE_SMOOTHING_SCALES_A = [
    0.5, 1.0, 2.0, 3.0, 5.0
]

# Scale used for the detailed side-by-side field figures.
PRIMARY_CHARGE_SCALE_A = 2.0

# Ignore almost-zero charge when evaluating sign agreement.
SIGN_THRESHOLD_NORMALIZED = 0.05

# Minimum smoothed occupancy used to consider a grid cell supported.
SMOOTHED_SUPPORT_THRESHOLD = 0.02

# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = Path(
    "/home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2"
)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Plane CSV      :", PLANE_POINTS_CSV)
print("Plane metadata :", PLANE_METADATA_JSON)
print("SURFMAP folder :", SURFMAP_OUTPUT_DIR)
print("Output folder  :", OUTPUT_DIR)


Plane CSV      : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_planar_projection_corrected/nylon_planar_projection_points.csv
Plane metadata : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_planar_projection_corrected/nylon_planar_projection_metadata.json
SURFMAP folder : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/output_SURFMAP_NYL_newVer_rotY90resnum_bfactor
Output folder  : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2



## 3. Input readers

### Why file discovery is conservative

A SURFMAP directory can contain multiple intermediate outputs. Accidentally combining a `coord_list` from one run with a shell from another run would create a completely artificial deformation.

Therefore, automatic discovery is accepted only when the choice is unambiguous. Otherwise the notebook stops and asks for the exact path.


In [5]:

def require_file(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )
    return path


def discover_candidates(folder: Path, patterns):
    found = []

    for pattern in patterns:
        found.extend(folder.rglob(pattern))

    unique = []
    seen = set()

    for path in found:
        resolved = str(path.resolve())

        if resolved not in seen:
            seen.add(resolved)
            unique.append(path)

    return unique


def choose_single_candidate(
    candidates,
    label,
    prefer_terms=(),
):
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No candidate found for {label}."
        )

    preferred = [
        p for p in candidates
        if all(
            term.lower() in p.name.lower()
            for term in prefer_terms
        )
    ]

    pool = (
        preferred
        if len(preferred) == 1
        else candidates
    )

    if len(pool) == 1:
        print(f"{label}: {pool[0]}")
        return pool[0]

    print(
        f"\nMultiple candidates found for {label}:"
    )

    for i, path in enumerate(candidates):
        print(f"  [{i}] {path}")

    raise RuntimeError(
        f"Set the exact {label} path manually "
        "in the settings cell."
    )


def read_pdb_xyz(path: Path):
    rows = []

    with path.open("r") as handle:
        for line in handle:
            if not line.startswith(
                ("ATOM", "HETATM")
            ):
                continue

            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                rows.append((x, y, z))

            except ValueError:
                continue

    df = pd.DataFrame(
        rows,
        columns=["x", "y", "z"],
    )

    if df.empty:
        raise ValueError(
            f"No ATOM/HETATM coordinates "
            f"found in {path}"
        )

    df.insert(
        0,
        "shell_index0",
        np.arange(len(df), dtype=int),
    )

    return df


def read_surfmap_coord_list(path: Path):
    """
    Expected semantic fields:
        absc, ord, value, resnb, restype, chain, index_sol

    Accepts either:
        - a headered whitespace table
        - a headerless whitespace table
    """

    df = pd.read_csv(
        path,
        sep=r"\s+",
        comment="#",
        engine="python",
    )

    lower_map = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    required = {
        "absc",
        "ord",
        "value",
        "index_sol",
    }

    if required.issubset(lower_map):
        rename = {
            lower_map["absc"]: "absc",
            lower_map["ord"]: "ord",
            lower_map["value"]: "value",
            lower_map["index_sol"]: "index_sol",
        }

        df = df.rename(columns=rename)

    else:
        df = pd.read_csv(
            path,
            sep=r"\s+",
            comment="#",
            header=None,
            engine="python",
        )

        if df.shape[1] < 7:
            raise ValueError(
                "coord_list has fewer than 7 columns. "
                "Inspect the format before continuing."
            )

        names = [
            "absc",
            "ord",
            "value",
            "resnb",
            "restype",
            "chain",
            "index_sol",
        ]

        extra = [
            f"extra_{i}"
            for i in range(
                df.shape[1] - len(names)
            )
        ]

        df.columns = names + extra

    for col in [
        "absc",
        "ord",
        "value",
        "index_sol",
    ]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    df = df.dropna(
        subset=[
            "absc",
            "ord",
            "value",
            "index_sol",
        ]
    ).copy()

    if not np.allclose(
        df["index_sol"],
        np.round(df["index_sol"]),
        atol=1e-8,
    ):
        raise ValueError(
            "index_sol contains non-integer values."
        )

    df["index_sol"] = np.round(
        df["index_sol"]
    ).astype(int)

    return df.reset_index(drop=True)


print("Input functions ready.")


Input functions ready.



# PART I — Build an exact point-to-point correspondence

## 4. Load the physical NYLON_PLANE representation

This is our geometric reference.

Every row corresponds to one real shell point and contains:

- original 3D coordinates;
- physical planar coordinates in Å;
- depth;
- charge.

The notebook refuses to continue if one of the required quantities is missing.


In [6]:

require_file(
    PLANE_POINTS_CSV,
    "Plane points CSV",
)

require_file(
    PLANE_METADATA_JSON,
    "Plane metadata JSON",
)

plane_df = pd.read_csv(
    PLANE_POINTS_CSV
)

with PLANE_METADATA_JSON.open("r") as f:
    plane_meta = json.load(f)

required_plane_columns = {
    "point_id",
    "x",
    "y",
    "z",
    "u_A",
    "v_A",
    "radial_distance_A",
    "depth_A",
    "charge_raw",
}

missing = (
    required_plane_columns
    - set(plane_df.columns)
)

if missing:
    raise ValueError(
        "Plane CSV is missing required columns: "
        + ", ".join(sorted(missing))
    )

print("Plane points:", len(plane_df))

print(
    "u range:",
    plane_df["u_A"].min(),
    "to",
    plane_df["u_A"].max(),
    "Å",
)

print(
    "v range:",
    plane_df["v_A"].min(),
    "to",
    plane_df["v_A"].max(),
    "Å",
)

print(
    "Depth range:",
    plane_df["depth_A"].min(),
    "to",
    plane_df["depth_A"].max(),
    "Å",
)

display(plane_df.head())


Plane points: 6148
u range: -24.09942945715506 to 24.09942945715506 Å
v range: -35.930700547418866 to 35.930700547418866 Å
Depth range: -8.881784197001252e-16 to 7.99972621437515 Å


,point_id,x,y,z,proj_x,proj_y,proj_z,u_A,v_A,radial_distance_A,signed_contact_distance_A,depth_A,charge_raw,charge_normalized
0,0,-3.827,16.363,36.591,-4.847709,16.329934,38.309322,-20.375985,-17.532701,26.880780,-1.998892,1.998892,-0.60,-0.769231
1,1,-3.596,15.683,37.554,-4.248477,15.661863,38.652418,-19.679039,-18.194047,26.800894,-1.277770,1.277770,0.06,0.076923
2,2,-3.612,16.251,36.962,-4.524969,16.221424,38.498946,-20.000617,-17.637542,26.666600,-1.787901,1.787901,-0.60,-0.769231
3,3,-3.649,15.361,37.344,-4.377119,15.337412,38.569759,-19.828658,-18.520028,27.132400,-1.425903,1.425903,0.06,0.076923
4,4,-3.883,16.024,36.370,-4.983254,15.988357,38.222233,-20.533633,-17.875890,27.224576,-2.154668,2.154668,-0.60,-0.769231



## 5. Load SURFMAP numerical coordinates and the corresponding shell

The PDF and PNG are not used for the primary geometric comparison.

The `coord_list` is used because it contains numerical 2D coordinates and an `index_sol` field that allows us to reconnect the 2D representation to the original 3D shell.


In [7]:

if SURFMAP_COORD_LIST is None:

    coord_candidates = discover_candidates(
        SURFMAP_OUTPUT_DIR,
        [
            "*coord_list*.txt",
            "*coord*list*.txt",
        ],
    )

    coord_list_path = choose_single_candidate(
        coord_candidates,
        "SURFMAP_COORD_LIST",
        prefer_terms=("bfactor",),
    )

else:
    coord_list_path = require_file(
        Path(SURFMAP_COORD_LIST),
        "SURFMAP coord_list",
    )


if SURFMAP_SHELL_PDB is None:

    metadata_shell = Path(
        plane_meta.get(
            "neutral_shell_path",
            "",
        )
    )

    if metadata_shell.exists():

        shell_path = metadata_shell

        print(
            "SURFMAP shell from plane metadata:",
            shell_path,
        )

    else:

        shell_candidates = discover_candidates(
            SURFMAP_OUTPUT_DIR,
            [
                "*mapped_shell*.pdb",
                "*shell*.pdb",
            ],
        )

        shell_path = choose_single_candidate(
            shell_candidates,
            "SURFMAP_SHELL_PDB",
            prefer_terms=("mapped_shell",),
        )

else:

    shell_path = require_file(
        Path(SURFMAP_SHELL_PDB),
        "SURFMAP shell PDB",
    )


coord_df = read_surfmap_coord_list(
    coord_list_path
)

surf_shell_df = read_pdb_xyz(
    shell_path
)

print()
print("coord_list rows :", len(coord_df))
print("shell PDB points:", len(surf_shell_df))

print(
    "SURFMAP absc range:",
    coord_df["absc"].min(),
    "to",
    coord_df["absc"].max(),
)

print(
    "SURFMAP ord range:",
    coord_df["ord"].min(),
    "to",
    coord_df["ord"].max(),
)

display(coord_df.head())


SURFMAP_COORD_LIST: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/output_SURFMAP_NYL_newVer_rotY90resnum_bfactor/coord_lists/NYL_newVer_rotY90resnum_bfactor_coord_list.txt
SURFMAP shell from plane metadata: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/output_SURFMAP_NYL_newVer_rotY90resnum_bfactor/NYL_newVer_rotY90resnum_bfactor_mapped_shell.pdb

coord_list rows : 26638
shell PDB points: 26638
SURFMAP absc range: 2 to 360
SURFMAP ord range: 2 to 180


,absc,ord,value,resnb,restype,chain,index_sol
0,2,86,0.36,837,NYL,A,26462
1,2,90,0.36,837,NYL,A,17571
2,2,92,0.06,828,NYL,Z,24799
3,2,96,0.06,828,NYL,Z,17570
4,4,80,0.10,837,NYL,A,24801



## 6. Determine whether `index_sol` is 0-based or 1-based

We do not assume the indexing convention.

The notebook tests both possibilities and asks:

> “Which convention reconnects the SURFMAP map points to the actual 3D shell coordinates?”

Using the wrong convention would shift every point index and could create a false deformation.

This check protects the comparison from that bookkeeping error.


In [8]:

def evaluate_index_offset(
    coord_df,
    shell_xyz,
    plane_xyz,
    offset,
    tolerance_A,
):

    indices0 = (
        coord_df["index_sol"]
        .to_numpy(dtype=int)
        - offset
    )

    valid = (
        (indices0 >= 0)
        & (indices0 < len(shell_xyz))
    )

    valid_indices = indices0[valid]

    xyz = shell_xyz[valid_indices]

    tree = cKDTree(plane_xyz)

    nearest_distance, _ = tree.query(
        xyz,
        k=1,
    )

    exact = (
        nearest_distance <= tolerance_A
    )

    return {
        "offset": offset,
        "valid_rows": int(valid.sum()),
        "matching_plane_points": int(
            exact.sum()
        ),
        "median_nearest_distance_A": float(
            np.median(nearest_distance)
        ),
        "mean_nearest_distance_A": float(
            np.mean(nearest_distance)
        ),
        "indices0": indices0,
    }


surf_shell_xyz = (
    surf_shell_df[
        ["x", "y", "z"]
    ]
    .to_numpy(dtype=float)
)

plane_xyz = (
    plane_df[
        ["x", "y", "z"]
    ]
    .to_numpy(dtype=float)
)

offset_tests = [
    evaluate_index_offset(
        coord_df,
        surf_shell_xyz,
        plane_xyz,
        offset=0,
        tolerance_A=XYZ_MATCH_TOL_A,
    ),
    evaluate_index_offset(
        coord_df,
        surf_shell_xyz,
        plane_xyz,
        offset=1,
        tolerance_A=XYZ_MATCH_TOL_A,
    ),
]

offset_table = pd.DataFrame(
    [
        {
            k: v
            for k, v in test.items()
            if k != "indices0"
        }
        for test in offset_tests
    ]
)

display(offset_table)

best_offset_test = sorted(
    offset_tests,
    key=lambda d: (
        -d["matching_plane_points"],
        d["median_nearest_distance_A"],
    ),
)[0]

INDEX_OFFSET = (
    best_offset_test["offset"]
)

print(
    "Selected index offset:",
    INDEX_OFFSET,
)


,offset,valid_rows,matching_plane_points,median_nearest_distance_A,mean_nearest_distance_A
0,0,26637,6147,20.528949,21.175710
1,1,26638,6148,20.525620,21.174915


Selected index offset: 1



## 7. Reconstruct the physical shell point behind each SURFMAP coordinate

After the indexing convention is known, `index_sol` is converted into the corresponding 3D shell coordinate.

Duplicate shell indices are checked explicitly.

A duplicate can appear if the same physical point is represented more than once around a map seam. In that case the representation closest to the map centre is retained for the main point-to-point comparison.


In [9]:

coord_work = coord_df.copy()

coord_work["shell_index0"] = (
    coord_work["index_sol"].astype(int)
    - INDEX_OFFSET
)

valid = (
    (coord_work["shell_index0"] >= 0)
    & (
        coord_work["shell_index0"]
        < len(surf_shell_df)
    )
)

if not valid.all():

    print(
        "Dropping invalid index_sol rows:",
        int((~valid).sum()),
    )

coord_work = (
    coord_work
    .loc[valid]
    .copy()
)

indices0 = (
    coord_work["shell_index0"]
    .to_numpy(dtype=int)
)

coord_work["shell_x"] = (
    surf_shell_xyz[indices0, 0]
)

coord_work["shell_y"] = (
    surf_shell_xyz[indices0, 1]
)

coord_work["shell_z"] = (
    surf_shell_xyz[indices0, 2]
)

duplicate_shell_rows = int(
    coord_work["shell_index0"]
    .duplicated(keep=False)
    .sum()
)

print(
    "Rows involved in duplicate shell indices:",
    duplicate_shell_rows,
)

if duplicate_shell_rows > 0:

    coord_work["_center_r2"] = (
        (
            coord_work["absc"]
            - SURFMAP_CENTER_2D[0]
        ) ** 2
        +
        (
            coord_work["ord"]
            - SURFMAP_CENTER_2D[1]
        ) ** 2
    )

    coord_work = (
        coord_work
        .sort_values("_center_r2")
        .drop_duplicates(
            subset="shell_index0",
            keep="first",
        )
        .drop(columns="_center_r2")
        .reset_index(drop=True)
    )

print(
    "Unique SURFMAP shell points:",
    len(coord_work),
)


Rows involved in duplicate shell indices: 0
Unique SURFMAP shell points: 26638



## 8. Match the same physical points in NYLON_PLANE and SURFMAP

The matching is based on the original **3D coordinates**, not on the 2D maps.

This is essential.

If we matched points according to their 2D position, we would already assume that the two projections are similar. Instead, the same physical point is identified first, and only afterwards are its two 2D positions compared.

For every accepted point we obtain:

```text
same physical shell point
        |
        +----> (u,v) in NYLON_PLANE
        |
        +----> (absc,ord) in SURFMAP
```

`Coverage` reports the fraction of NYLON_PLANE points that could be identified in SURFMAP.


In [10]:

surf_xyz_unique = (
    coord_work[
        [
            "shell_x",
            "shell_y",
            "shell_z",
        ]
    ]
    .to_numpy(dtype=float)
)

surf_tree = cKDTree(
    surf_xyz_unique
)

match_dist, surf_row_idx = (
    surf_tree.query(
        plane_xyz,
        k=1,
    )
)

matched_mask = (
    match_dist <= XYZ_MATCH_TOL_A
)

coverage = matched_mask.mean()

print(
    "Plane points total:",
    len(plane_df),
)

print(
    "Matched points:",
    int(matched_mask.sum()),
)

print(
    "Coverage:",
    f"{100 * coverage:.2f}%",
)

if matched_mask.any():

    print(
        "Maximum accepted XYZ mismatch:",
        match_dist[matched_mask].max(),
        "Å",
    )

if coverage < 0.90:

    warnings.warn(
        "Coverage below 90%. Verify that "
        "the SURFMAP coord_list and shell "
        "belong to the same run."
    )


plane_match = (
    plane_df
    .loc[matched_mask]
    .reset_index(drop=True)
    .copy()
)

surf_match = (
    coord_work
    .iloc[
        surf_row_idx[matched_mask]
    ]
    .reset_index(drop=True)
    .copy()
)

if (
    surf_match["shell_index0"]
    .duplicated()
    .any()
):

    raise ValueError(
        "The same SURFMAP shell point "
        "matched more than one plane point."
    )


matched = pd.DataFrame({
    "point_id":
        plane_match["point_id"],

    "x":
        plane_match["x"],

    "y":
        plane_match["y"],

    "z":
        plane_match["z"],

    "u_A":
        plane_match["u_A"],

    "v_A":
        plane_match["v_A"],

    "plane_radius_A":
        plane_match[
            "radial_distance_A"
        ],

    "depth_A":
        plane_match["depth_A"],

    "charge_raw":
        plane_match["charge_raw"],

    "surf_absc":
        surf_match["absc"],

    "surf_ord":
        surf_match["ord"],

    "surf_value":
        surf_match["value"],

    "surf_shell_index0":
        surf_match[
            "shell_index0"
        ],

    "xyz_match_error_A":
        match_dist[matched_mask],
})

display(matched.head())


Plane points total: 6148
Matched points: 6148
Coverage: 100.00%
Maximum accepted XYZ mismatch: 0.0 Å


,point_id,x,y,z,u_A,v_A,plane_radius_A,depth_A,charge_raw,surf_absc,surf_ord,surf_value,surf_shell_index0,xyz_match_error_A
0,0,-3.827,16.363,36.591,-20.375985,-17.532701,26.880780,1.998892,-0.60,216,122,-0.60,0,0.0
1,1,-3.596,15.683,37.554,-19.679039,-18.194047,26.800894,1.277770,0.06,218,124,0.06,1,0.0
2,2,-3.612,16.251,36.962,-20.000617,-17.637542,26.666600,1.787901,-0.60,216,124,-0.60,2,0.0
3,3,-3.649,15.361,37.344,-19.828658,-18.520028,27.132400,1.425903,0.06,218,124,0.06,3,0.0
4,4,-3.883,16.024,36.370,-20.533633,-17.875890,27.224576,2.154668,-0.60,218,122,-0.60,4,0.0



# PART II — Is the SURFMAP coordinate set already discretized?

## 9. Quantization / coordinate-collision diagnostics

This check is critical for interpreting fine-scale metrics.

If `absc` and `ord` are continuous coordinates, neighbouring shell points should usually have distinct floating-point positions.

If instead:

- almost all coordinates are integer-valued;
- many different physical points share the same `(absc,ord)`;

then the available `coord_list` is already a **quantized map representation**.

That means a local point collapse can be caused by the map grid even if the underlying continuous spherical transformation did not literally send the points to exactly the same mathematical location.

The notebook therefore reports this explicitly rather than silently attributing all local collapse to spherical geometry.


In [11]:

S_raw = matched[
    ["surf_absc", "surf_ord"]
].to_numpy(dtype=float)

P = matched[
    ["u_A", "v_A"]
].to_numpy(dtype=float)

integer_like = np.isclose(
    S_raw,
    np.round(S_raw),
    atol=1e-10,
)

integer_fraction = float(
    integer_like.mean()
)

surface_coord_table = pd.DataFrame(
    S_raw,
    columns=[
        "surf_absc",
        "surf_ord",
    ],
)

multiplicity = (
    surface_coord_table
    .value_counts()
)

n_unique_surf_positions = int(
    len(multiplicity)
)

n_total_points = int(
    len(surface_coord_table)
)

n_points_in_shared_positions = int(
    multiplicity[
        multiplicity > 1
    ].sum()
)

shared_position_fraction = (
    n_points_in_shared_positions
    / n_total_points
)

max_position_multiplicity = int(
    multiplicity.max()
)


def smallest_positive_step(values):
    unique = np.sort(
        np.unique(
            np.asarray(values, dtype=float)
        )
    )

    if len(unique) < 2:
        return np.nan

    diffs = np.diff(unique)

    diffs = diffs[
        diffs > 1e-12
    ]

    if len(diffs) == 0:
        return np.nan

    return float(
        np.min(diffs)
    )


surf_step_x = smallest_positive_step(
    S_raw[:, 0]
)

surf_step_y = smallest_positive_step(
    S_raw[:, 1]
)

print(
    "Fraction of SURFMAP coordinates "
    "that are integer-like:",
    f"{100 * integer_fraction:.2f}%"
)

print(
    "Matched physical points:",
    n_total_points,
)

print(
    "Unique SURFMAP 2D positions:",
    n_unique_surf_positions,
)

print(
    "Unique-position fraction:",
    f"{100 * n_unique_surf_positions / n_total_points:.2f}%"
)

print(
    "Points belonging to a shared "
    "SURFMAP position:",
    n_points_in_shared_positions,
    f"({100 * shared_position_fraction:.2f}%)",
)

print(
    "Maximum physical points sharing "
    "one SURFMAP coordinate:",
    max_position_multiplicity,
)

print(
    "Smallest positive SURFMAP x step:",
    surf_step_x,
)

print(
    "Smallest positive SURFMAP y step:",
    surf_step_y,
)

SURFMAP_COORDS_LOOK_QUANTIZED = (
    integer_fraction > 0.99
    and shared_position_fraction > 0.05
)

print()
print(
    "SURFMAP coordinates look quantized/grid-based:",
    SURFMAP_COORDS_LOOK_QUANTIZED,
)


Fraction of SURFMAP coordinates that are integer-like: 100.00%
Matched physical points: 6148
Unique SURFMAP 2D positions: 1972
Unique-position fraction: 32.08%
Points belonging to a shared SURFMAP position: 5652 (91.93%)
Maximum physical points sharing one SURFMAP coordinate: 21
Smallest positive SURFMAP x step: 2.0
Smallest positive SURFMAP y step: 2.0

SURFMAP coordinates look quantized/grid-based: True



# PART III — What does “aligned” mean?

## 10. Similarity alignment explained step by step

The two maps cannot be compared by directly subtracting their coordinates because they use different coordinate systems.

For example, the same rectangle could be represented as:

```text
Map A                 Map B

+---------+                +-----+
|         |               /     /
|         |              /     /
+---------+             +-----+
```

Some differences can be completely harmless:

- Map B may use a different origin;
- it may be rotated;
- it may be uniformly larger or smaller;
- the axis convention may be mirrored.

We should not count those as deformation.

### Step 1 — Centre both point clouds

Let:

\[
S_i
\]

be the SURFMAP coordinates and

\[
P_i
\]

the corresponding planar coordinates.

We subtract the centroid from each map:

\[
S_i^0 = S_i-\bar S
\]

\[
P_i^0 = P_i-\bar P
\]

Now both clouds are centred around `(0,0)`.

### Step 2 — Find the best rotation or reflection

Because point correspondence is already known, we can ask:

> Which rigid orientation of the SURFMAP cloud gives the smallest total squared distance from the planar cloud?

This is the 2D **orthogonal Procrustes** problem.

It is solved using singular value decomposition (SVD) of the cross-covariance matrix:

\[
C=(S^0)^T P^0
\]

The resulting orthogonal matrix \(R\) rotates the SURFMAP point cloud. If its determinant is `-1`, a mirror reflection is also present. If the determinant is `+1`, the transformation is a pure rotation.

### Step 3 — Find one global scale factor

After orientation is corrected, we find a single number \(s\) such that:

\[
s(S^0R)
\]

best matches \(P^0\).

Only **one** scale is allowed for both axes.

This is important: a uniform zoom does not change shape, while separate x/y scaling would hide real anisotropic deformation.

### Step 4 — Restore the planar centre

Finally:

\[
S_{aligned}
=
s S R + t
\]

where \(t\) is the translation that places the aligned SURFMAP cloud in the planar coordinate frame.

### What alignment is NOT allowed to do

It cannot:

- straighten a curve;
- stretch only one side;
- scale x and y differently;
- move individual points;
- perform a nonlinear warp.

Therefore, if the SURFMAP map remains curved after similarity alignment, that curvature is a genuine shape difference between the two representations.

A useful mental model is:

> **Alignment is allowed to move a transparent sheet, rotate it and zoom it uniformly. It is not allowed to bend the sheet.**


In [12]:

def fit_similarity_transform(
    source,
    target,
    allow_reflection=True,
):
    """
    Fit:
        target ≈ scale * source @ R + translation

    where R is orthogonal.

    Because point correspondence is fixed, this
    transformation cannot rearrange individual points.
    """

    X = np.asarray(
        source,
        dtype=float,
    )

    Y = np.asarray(
        target,
        dtype=float,
    )

    X_mean = X.mean(axis=0)
    Y_mean = Y.mean(axis=0)

    X0 = X - X_mean
    Y0 = Y - Y_mean

    cross_covariance = (
        X0.T @ Y0
    )

    U, _, Vt = np.linalg.svd(
        cross_covariance
    )

    R_best_orthogonal = (
        U @ Vt
    )

    candidates = []

    def build_candidate(
        R,
        label,
    ):
        rotated = X0 @ R

        scale = (
            np.sum(rotated * Y0)
            / np.sum(rotated ** 2)
        )

        aligned = (
            scale * rotated
            + Y_mean
        )

        residual = (
            aligned - Y
        )

        return {
            "label": label,
            "R": R,
            "scale": float(scale),
            "source_centroid": X_mean,
            "target_centroid": Y_mean,
            "aligned": aligned,
            "residual": residual,
            "sse": float(
                np.sum(residual ** 2)
            ),
            "detR": float(
                np.linalg.det(R)
            ),
        }

    det_any = np.linalg.det(
        R_best_orthogonal
    )

    label_any = (
        "rotation"
        if det_any > 0
        else "rotation + reflection"
    )

    candidates.append(
        build_candidate(
            R_best_orthogonal,
            label_any,
        )
    )

    if allow_reflection:

        # Also explicitly evaluate the alternative
        # handedness so the notebook can report
        # whether reflection was actually useful.
        U_alt = U.copy()

        U_alt[:, -1] *= -1

        R_alt = (
            U_alt @ Vt
        )

        det_alt = np.linalg.det(
            R_alt
        )

        label_alt = (
            "rotation"
            if det_alt > 0
            else "rotation + reflection"
        )

        candidates.append(
            build_candidate(
                R_alt,
                label_alt,
            )
        )

    else:

        if (
            np.linalg.det(
                R_best_orthogonal
            ) < 0
        ):

            U_rot = U.copy()

            U_rot[:, -1] *= -1

            R_rot = (
                U_rot @ Vt
            )

            candidates = [
                build_candidate(
                    R_rot,
                    "rotation",
                )
            ]

    best = min(
        candidates,
        key=lambda x: x["sse"],
    )

    return best, candidates


similarity_fit, similarity_candidates = (
    fit_similarity_transform(
        S_raw,
        P,
        allow_reflection=True,
    )
)

S_aligned = (
    similarity_fit["aligned"]
)

residual_xy = (
    S_aligned - P
)

residual_A = np.linalg.norm(
    residual_xy,
    axis=1,
)

similarity_rms_A = float(
    np.sqrt(
        np.mean(
            residual_A ** 2
        )
    )
)

similarity_p95_A = float(
    np.percentile(
        residual_A,
        95,
    )
)

similarity_max_A = float(
    np.max(residual_A)
)

P_centered = (
    P - P.mean(axis=0)
)

rms_radius_A = float(
    np.sqrt(
        np.mean(
            np.sum(
                P_centered ** 2,
                axis=1,
            )
        )
    )
)

bbox_diag_A = float(
    np.linalg.norm(
        [
            P[:, 0].max()
            - P[:, 0].min(),

            P[:, 1].max()
            - P[:, 1].min(),
        ]
    )
)

nrms_rms_radius_pct = (
    100
    * similarity_rms_A
    / rms_radius_A
)

nrms_bbox_diag_pct = (
    100
    * similarity_rms_A
    / bbox_diag_A
)

procrustes_disparity = float(
    np.sum(residual_A ** 2)
    /
    np.sum(
        np.sum(
            P_centered ** 2,
            axis=1,
        )
    )
)

R = similarity_fit["R"]
detR = similarity_fit["detR"]

if detR > 0:

    rotation_angle_deg = float(
        np.degrees(
            np.arctan2(
                R[0, 1],
                R[0, 0],
            )
        )
    )

else:

    rotation_angle_deg = np.nan


matched[
    "surf_aligned_u_A"
] = S_aligned[:, 0]

matched[
    "surf_aligned_v_A"
] = S_aligned[:, 1]

matched[
    "positional_error_A"
] = residual_A


print(
    "Selected alignment:",
    similarity_fit["label"],
)

print(
    "det(R):",
    detR,
)

if detR > 0:

    print(
        "Approximate rotation angle:",
        rotation_angle_deg,
        "degrees",
    )

else:

    print(
        "A reflection is part of the "
        "best alignment."
    )

print(
    "Uniform scale factor:",
    similarity_fit["scale"],
)

print(
    "SURFMAP centroid:",
    similarity_fit[
        "source_centroid"
    ],
)

print(
    "Plane centroid:",
    similarity_fit[
        "target_centroid"
    ],
)

print()
print(
    "RMS positional deformation:",
    similarity_rms_A,
    "Å",
)

print(
    "95th percentile deformation:",
    similarity_p95_A,
    "Å",
)

print(
    "Maximum deformation:",
    similarity_max_A,
    "Å",
)

print(
    "NRMSD / RMS radius:",
    nrms_rms_radius_pct,
    "%",
)

print(
    "NRMSD / map diagonal:",
    nrms_bbox_diag_pct,
    "%",
)

print(
    "Procrustes disparity:",
    procrustes_disparity,
)


Selected alignment: rotation
det(R): 1.0000000000000002
Approximate rotation angle: -89.026824790831 degrees
Uniform scale factor: 0.36477116259494896
SURFMAP centroid: [181.48048146 139.71340273]
Plane centroid: [ 0.34211849 -0.0910969 ]

RMS positional deformation: 15.286261175965159 Å
95th percentile deformation: 24.7759644484104 Å
Maximum deformation: 37.495154791567295 Å
NRMSD / RMS radius: 58.58796442502555 %
NRMSD / map diagonal: 17.66614620518428 %
Procrustes disparity: 0.3432549575468059



## 11. How large is one SURFMAP grid step after alignment?

If the `coord_list` is quantized, it is useful to estimate the physical size of one map-coordinate step after the global scale conversion.

For a uniform alignment scale \(s\):

\[
1\ \text{SURFMAP coordinate unit}
\approx
|s|\ \text{Å}
\]

if the raw grid step is 1.

This lets us compare:

- error that could plausibly arise from coordinate quantization;
- the much larger global shape deformation.

This does **not** fully reconstruct the unknown SURFMAP binning rule. It is only a scale diagnostic.


In [13]:

minimum_raw_grid_step = np.nanmin(
    [
        surf_step_x,
        surf_step_y,
    ]
)

effective_grid_step_A = (
    abs(
        similarity_fit["scale"]
    )
    * minimum_raw_grid_step
)

print(
    "Smallest raw SURFMAP coordinate step:",
    minimum_raw_grid_step,
)

print(
    "Approximate physical size after alignment:",
    effective_grid_step_A,
    "Å",
)

print(
    "Global RMS error / one grid step:",
    similarity_rms_A
    / effective_grid_step_A
    if effective_grid_step_A > 0
    else np.nan,
)

if (
    SURFMAP_COORDS_LOOK_QUANTIZED
):

    print()
    print(
        "Interpretation: fine-scale local "
        "metrics are affected by grid "
        "collisions, but the global RMS "
        "error can be compared with the "
        "physical grid step to determine "
        "whether quantization alone could "
        "explain the observed deformation."
    )


Smallest raw SURFMAP coordinate step: 2.0
Approximate physical size after alignment: 0.7295423251898979 Å
Global RMS error / one grid step: 20.95322046186448

Interpretation: fine-scale local metrics are affected by grid collisions, but the global RMS error can be compared with the physical grid step to determine whether quantization alone could explain the observed deformation.



## 12. Alignment figure for colleagues

The left panel shows the two coordinate sets in their own original coordinate systems.

The right panel shows the **only** comparison that should be interpreted geometrically: SURFMAP after the best similarity alignment against the physical planar reference.

If the right-hand shapes still differ strongly, that difference cannot be removed by translation, rotation, reflection or one uniform scale.


In [14]:

fig_alignment = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Original coordinate systems",
        "After similarity alignment",
    ),
)

fig_alignment.add_trace(
    go.Scattergl(
        x=P[:, 0],
        y=P[:, 1],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.55,
        ),
        name="NYLON_PLANE",
        legendgroup="plane",
    ),
    row=1,
    col=1,
)

fig_alignment.add_trace(
    go.Scattergl(
        x=S_raw[:, 0],
        y=S_raw[:, 1],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.55,
        ),
        name="SURFMAP raw coords",
        legendgroup="surf",
    ),
    row=1,
    col=1,
)

fig_alignment.add_trace(
    go.Scattergl(
        x=P[:, 0],
        y=P[:, 1],
        mode="markers",
        marker=dict(
            size=4,
            opacity=0.50,
        ),
        name="NYLON_PLANE",
        legendgroup="plane",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig_alignment.add_trace(
    go.Scattergl(
        x=S_aligned[:, 0],
        y=S_aligned[:, 1],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.50,
        ),
        name="SURFMAP aligned",
        legendgroup="surf",
    ),
    row=1,
    col=2,
)

fig_alignment.update_xaxes(
    scaleanchor="y",
    scaleratio=1,
    row=1,
    col=1,
)

fig_alignment.update_xaxes(
    scaleanchor="y2",
    scaleratio=1,
    row=1,
    col=2,
)

fig_alignment.update_layout(
    title=(
        "What alignment removes — and "
        "what deformation remains"
    ),
    template="plotly_white",
    width=1250,
    height=620,
)

fig_alignment.show()



# PART IV — Global geometric deformation

## 13. Where is the SURFMAP positional error located?

For every corresponding physical shell point we calculate:

\[
e_i =
\left\|
S_{i,aligned} - P_i
\right\|
\]

where \(P_i\) is the point position in **NYLON_PLANE** and \(S_{i,aligned}\) is the position assigned to the same physical point by SURFMAP after similarity alignment.

The value \(e_i\) is therefore a **2D positional error in Å**.

It is important not to confuse this quantity with `depth`:

- `depth` = how far the original 3D shell point lies behind the contact plane;
- `positional error` = how far the SURFMAP 2D position of that point is from its physical planar 2D position.

### Why the same error is shown twice

The same pointwise error value is displayed in two coordinate systems.

**Left — NYLON_PLANE**

The point is plotted at its physical planar position `(u,v)`.

This answers:

> **Which physical regions of the nylon face are represented most poorly by SURFMAP?**

**Right — SURFMAP aligned**

The same physical point is plotted at the position assigned by SURFMAP after global similarity alignment.

This answers:

> **Which regions of the SURFMAP map itself correspond to the largest geometrical error?**

The colour value is exactly the same in both panels. Only the position used for visualization changes.

This second view is particularly useful for SURFMATCH because it shows directly which parts of the SURFMAP representation would be geometrically less reliable if that map were used for matching.

For example, `positional error = 5 Å` does **not** mean that the point is 5 Å deep. It means that, after removing translation, rotation and global scale differences, the SURFMAP position of that physical shell point is still 5 Å away from its position in the planar reference.


In [15]:

error_min = 0.0
error_max = float(np.max(residual_A))

fig_error_comparison = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "A — Error shown on NYLON_PLANE",
        "B — Same error shown on SURFMAP aligned",
    ),
    horizontal_spacing=0.08,
)

# Panel A — physical planar reference
fig_error_comparison.add_trace(
    go.Scattergl(
        x=P[:, 0],
        y=P[:, 1],
        mode="markers",
        marker=dict(
            size=5,
            color=residual_A,
            colorscale="Viridis",
            cmin=error_min,
            cmax=error_max,
            showscale=False,
            line=dict(width=0),
        ),
        customdata=np.column_stack([
            matched["point_id"],
            residual_A,
            matched["plane_radius_A"],
            matched["depth_A"],
            S_aligned[:, 0],
            S_aligned[:, 1],
        ]),
        hovertemplate=(
            "point_id = %{customdata[0]:.0f}<br>"
            "PLANAR u = %{x:.3f} Å<br>"
            "PLANAR v = %{y:.3f} Å<br>"
            "SURFMAP aligned u = %{customdata[4]:.3f} Å<br>"
            "SURFMAP aligned v = %{customdata[5]:.3f} Å<br>"
            "<b>positional error = %{customdata[1]:.4f} Å</b><br>"
            "plane radius = %{customdata[2]:.3f} Å<br>"
            "depth = %{customdata[3]:.3f} Å"
            "<extra></extra>"
        ),
        name="Error on NYLON_PLANE",
    ),
    row=1,
    col=1,
)

# Panel B — aligned SURFMAP representation
fig_error_comparison.add_trace(
    go.Scattergl(
        x=S_aligned[:, 0],
        y=S_aligned[:, 1],
        mode="markers",
        marker=dict(
            size=5,
            color=residual_A,
            colorscale="Viridis",
            cmin=error_min,
            cmax=error_max,
            colorbar=dict(
                title="Positional error (Å)",
                len=0.80,
            ),
            line=dict(width=0),
        ),
        customdata=np.column_stack([
            matched["point_id"],
            residual_A,
            P[:, 0],
            P[:, 1],
            matched["plane_radius_A"],
            matched["depth_A"],
        ]),
        hovertemplate=(
            "point_id = %{customdata[0]:.0f}<br>"
            "SURFMAP aligned u = %{x:.3f} Å<br>"
            "SURFMAP aligned v = %{y:.3f} Å<br>"
            "PLANAR u = %{customdata[2]:.3f} Å<br>"
            "PLANAR v = %{customdata[3]:.3f} Å<br>"
            "<b>positional error = %{customdata[1]:.4f} Å</b><br>"
            "plane radius = %{customdata[4]:.3f} Å<br>"
            "depth = %{customdata[5]:.3f} Å"
            "<extra></extra>"
        ),
        name="Error on SURFMAP aligned",
    ),
    row=1,
    col=2,
)

fig_error_comparison.update_xaxes(
    title_text="u (Å)",
    scaleanchor="y",
    scaleratio=1,
    row=1,
    col=1,
)
fig_error_comparison.update_yaxes(
    title_text="v (Å)",
    row=1,
    col=1,
)
fig_error_comparison.update_xaxes(
    title_text="u-equivalent after alignment (Å)",
    scaleanchor="y2",
    scaleratio=1,
    row=1,
    col=2,
)
fig_error_comparison.update_yaxes(
    title_text="v-equivalent after alignment (Å)",
    row=1,
    col=2,
)

fig_error_comparison.update_layout(
    title=(
        "Spatial distribution of SURFMAP positional error — "
        "the same pointwise error shown in both coordinate systems"
    ),
    template="plotly_white",
    width=1250,
    height=650,
    margin=dict(l=60, r=120, t=100, b=60),
)

fig_error_comparison.show()

# Alias retained for the export section.
fig_error = fig_error_comparison



# PART V — Can a central region be represented better?

## 14. Corrected central-region analysis

The previous notebook grouped residuals by radius **after fitting one transformation to the entire surface**.

That is useful for describing where a globally aligned map differs, but it does **not** answer the stronger question:

> “If SURFMATCH used only a central region, how accurately could SURFMAP represent that central region by itself?”

A global fit is a compromise over the full surface. Strong deformation at the edges can influence the best rotation and scale and therefore make the centre appear worse than it would under its own optimal alignment.

### Correct strategy used here

For each physical radius \(R\):

1. select only NYLON_PLANE points with:
   \[
   \sqrt{u^2+v^2}\le R
   \]
2. take the corresponding SURFMAP points;
3. fit a **new similarity transformation using only those central points**;
4. calculate RMS and P95 residuals inside that region.

This still does **not** define a SURFMAP pixel ROI.

It simply measures the best geometric fidelity achievable for progressively larger physical central regions.


In [16]:

central_rows = []

plane_radius = matched[
    "plane_radius_A"
].to_numpy(dtype=float)

for radius_limit in CENTRAL_RADII_A:

    mask = (
        plane_radius
        <= radius_limit
    )

    n_points = int(
        mask.sum()
    )

    if n_points < 3:
        continue

    local_fit, _ = (
        fit_similarity_transform(
            S_raw[mask],
            P[mask],
            allow_reflection=True,
        )
    )

    local_error = np.linalg.norm(
        local_fit["aligned"]
        - P[mask],
        axis=1,
    )

    local_rms = float(
        np.sqrt(
            np.mean(
                local_error ** 2
            )
        )
    )

    local_p95 = float(
        np.percentile(
            local_error,
            95,
        )
    )

    P_local_centered = (
        P[mask]
        - P[mask].mean(axis=0)
    )

    local_rms_radius = float(
        np.sqrt(
            np.mean(
                np.sum(
                    P_local_centered ** 2,
                    axis=1,
                )
            )
        )
    )

    local_nrms_pct = (
        100
        * local_rms
        / local_rms_radius
    )

    central_rows.append({
        "physical_radius_A":
            float(radius_limit),

        "n_points":
            n_points,

        "best_fit_RMS_A":
            local_rms,

        "best_fit_P95_A":
            local_p95,

        "best_fit_NRMS_RMSradius_pct":
            local_nrms_pct,

        "best_fit_uniform_scale":
            float(
                local_fit["scale"]
            ),

        "detR":
            float(
                local_fit["detR"]
            ),
    })


central_region_df = pd.DataFrame(
    central_rows
)

display(central_region_df)


fig_central = go.Figure()

fig_central.add_trace(
    go.Scatter(
        x=central_region_df[
            "physical_radius_A"
        ],
        y=central_region_df[
            "best_fit_RMS_A"
        ],
        mode="lines+markers",
        name="Best-fit RMS",
    )
)

fig_central.add_trace(
    go.Scatter(
        x=central_region_df[
            "physical_radius_A"
        ],
        y=central_region_df[
            "best_fit_P95_A"
        ],
        mode="lines+markers",
        name="Best-fit P95",
    )
)

fig_central.update_layout(
    title=(
        "Best achievable SURFMAP "
        "geometric error for central "
        "physical regions"
    ),
    xaxis_title=(
        "Physical planar radius (Å)"
    ),
    yaxis_title=(
        "Residual error after local "
        "similarity fit (Å)"
    ),
    template="plotly_white",
)

fig_central.show()


fig_central_scale = go.Figure(
    go.Scatter(
        x=central_region_df[
            "physical_radius_A"
        ],
        y=central_region_df[
            "best_fit_uniform_scale"
        ],
        mode="lines+markers",
    )
)

fig_central_scale.update_layout(
    title=(
        "Best-fit SURFMAP scale "
        "as central region size increases"
    ),
    xaxis_title=(
        "Physical planar radius (Å)"
    ),
    yaxis_title=(
        "Uniform SURFMAP→plane scale"
    ),
    template="plotly_white",
)

fig_central_scale.show()


,physical_radius_A,n_points,best_fit_RMS_A,best_fit_P95_A,best_fit_NRMS_RMSradius_pct,best_fit_uniform_scale,detR
0,5.0,119,1.834703,3.453896,52.227014,0.367266,1.0
1,10.0,454,4.608290,9.167036,64.685941,0.293469,1.0
2,15.0,990,7.204272,13.795654,68.276648,0.279887,1.0
3,20.0,1780,9.610422,18.236101,67.864199,0.288811,1.0
4,25.0,3143,13.586627,25.981555,74.285134,0.268123,1.0
5,30.0,4302,14.403699,25.463078,68.157236,0.304693,1.0
6,35.0,5194,14.816009,25.018739,63.130942,0.335939,1.0
7,40.0,6080,15.207996,24.844550,58.755964,0.363181,1.0
8,45.0,6148,15.286261,24.775964,58.587964,0.364771,1.0



# PART VI — Multiscale local geometric distortion

## 15. Why nearest-neighbour stretch alone is not enough

If SURFMAP coordinates are already quantized, two very close physical shell points may fall into the same SURFMAP grid position.

For example:

```text
NYLON_PLANE

•  •      distance = 0.20 Å

SURFMAP grid

•         both points in the same map cell
```

The apparent SURFMAP distance becomes zero.

This is a real loss of spatial resolution in the available SURFMAP representation, but it should not be confused with the deformation of a larger 5–10 Å electrostatic patch.

### Corrected approach

We therefore calculate local distance distortion in several physical distance bands.

For the same physical point pair:

\[
R_d =
\frac{d_{SURFMAP,aligned}}
     {d_{PLANE}}
\]

and:

\[
error_{\%}
=
100|R_d-1|
\]

Interpretation:

```text
R_d = 1.0  -> distance preserved
R_d = 0.7  -> 30% compression
R_d = 1.3  -> 30% expansion
```

We also report the percentage of pairs that collapse to exactly the same SURFMAP coordinate.

Fine-scale bands are expected to be most sensitive to coordinate quantization. Distortion that remains large at 5–10 Å or 10–15 Å cannot be explained simply by sub-pixel point collapse.


In [17]:

def build_knn_edges(
    points,
    k,
):
    """
    Build a deterministic set of local point pairs.

    Using k nearest neighbours avoids constructing
    every possible N(N-1)/2 pair while still sampling
    multiple local physical scales.
    """

    tree = cKDTree(points)

    _, neighbour_idx = tree.query(
        points,
        k=min(
            k + 1,
            len(points),
        ),
    )

    edges = set()

    for i in range(
        len(points)
    ):

        for j in neighbour_idx[
            i,
            1:,
        ]:

            a, b = sorted(
                (
                    int(i),
                    int(j),
                )
            )

            if a != b:
                edges.add(
                    (a, b)
                )

    return np.asarray(
        sorted(edges),
        dtype=int,
    )


pair_edges = build_knn_edges(
    P,
    PAIR_K_NEIGHBORS,
)

edge_i = pair_edges[:, 0]
edge_j = pair_edges[:, 1]

d_plane_all = np.linalg.norm(
    P[edge_i] - P[edge_j],
    axis=1,
)

d_surf_all = np.linalg.norm(
    S_aligned[edge_i]
    - S_aligned[edge_j],
    axis=1,
)

multiscale_rows = []

for low_A, high_A in DISTANCE_BANDS_A:

    mask = (
        (d_plane_all > low_A)
        & (d_plane_all <= high_A)
    )

    n_pairs = int(
        mask.sum()
    )

    if n_pairs == 0:
        continue

    d_plane_band = (
        d_plane_all[mask]
    )

    d_surf_band = (
        d_surf_all[mask]
    )

    ratio = (
        d_surf_band
        / d_plane_band
    )

    absolute_error_pct = (
        100
        * np.abs(
            ratio - 1
        )
    )

    collapsed = (
        d_surf_band
        <= 1e-12
    )

    multiscale_rows.append({
        "distance_min_A":
            float(low_A),

        "distance_max_A":
            float(high_A),

        "n_pairs":
            n_pairs,

        "median_distance_ratio":
            float(
                np.median(ratio)
            ),

        "median_abs_distance_error_pct":
            float(
                np.median(
                    absolute_error_pct
                )
            ),

        "p95_abs_distance_error_pct":
            float(
                np.percentile(
                    absolute_error_pct,
                    95,
                )
            ),

        "collapsed_pair_pct":
            float(
                100
                * collapsed.mean()
            ),
    })


multiscale_geometry_df = (
    pd.DataFrame(
        multiscale_rows
    )
)

display(
    multiscale_geometry_df
)


band_labels = [
    (
        f"{row.distance_min_A:g}"
        f"–"
        f"{row.distance_max_A:g} Å"
    )
    for row
    in multiscale_geometry_df.itertuples()
]

fig_multiscale_geom = go.Figure()

fig_multiscale_geom.add_trace(
    go.Bar(
        x=band_labels,
        y=multiscale_geometry_df[
            "median_abs_distance_error_pct"
        ],
        name="Median absolute error",
    )
)

fig_multiscale_geom.add_trace(
    go.Bar(
        x=band_labels,
        y=multiscale_geometry_df[
            "p95_abs_distance_error_pct"
        ],
        name="95th percentile",
    )
)

fig_multiscale_geom.update_layout(
    title=(
        "Distance distortion at different "
        "physical length scales"
    ),
    xaxis_title=(
        "Physical pair-distance band "
        "in NYLON_PLANE"
    ),
    yaxis_title=(
        "Absolute distance distortion (%)"
    ),
    barmode="group",
    template="plotly_white",
)

fig_multiscale_geom.show()


fig_collapse = go.Figure(
    go.Bar(
        x=band_labels,
        y=multiscale_geometry_df[
            "collapsed_pair_pct"
        ],
    )
)

fig_collapse.update_layout(
    title=(
        "Pair collapse into the same "
        "SURFMAP 2D position"
    ),
    xaxis_title=(
        "Physical pair-distance band"
    ),
    yaxis_title=(
        "Collapsed pairs (%)"
    ),
    template="plotly_white",
)

fig_collapse.show()


,distance_min_A,distance_max_A,n_pairs,median_distance_ratio,median_abs_distance_error_pct,p95_abs_distance_error_pct,collapsed_pair_pct
0,0.0,0.5,6427,1.478259,100.000000,1338.226616,49.416524
1,0.5,1.0,15709,1.071075,39.067737,463.980233,19.523840
2,1.0,2.0,57104,0.841233,40.163195,212.791038,6.728075
3,2.0,5.0,322663,0.705300,37.980466,85.159003,0.555378
4,5.0,10.0,957783,0.652881,38.970142,80.436422,0.000940
5,10.0,15.0,305350,0.554648,45.904410,77.845472,0.000000



# PART VII — Charge-value consistency

## 16. Verify that the same point carries the same scalar value

Before comparing spatial electrostatic patterns, we verify that SURFMAP did not silently replace the point values.

This is a file/data consistency check.

If `charge_raw` and SURFMAP `value` are identical or linearly related, then later differences in the 2D charge maps can be attributed to spatial rearrangement rather than to different underlying point charges.


In [18]:

q_plane = matched[
    "charge_raw"
].to_numpy(dtype=float)

q_surf_value = matched[
    "surf_value"
].to_numpy(dtype=float)

finite_q = (
    np.isfinite(q_plane)
    & np.isfinite(q_surf_value)
)

if finite_q.sum() < 3:

    raise ValueError(
        "Not enough finite charge values."
    )

charge_value_pearson = float(
    pearsonr(
        q_plane[finite_q],
        q_surf_value[finite_q],
    ).statistic
)

A = np.column_stack([
    q_plane[finite_q],
    np.ones(
        finite_q.sum()
    ),
])

coef, *_ = np.linalg.lstsq(
    A,
    q_surf_value[finite_q],
    rcond=None,
)

charge_slope = float(
    coef[0]
)

charge_intercept = float(
    coef[1]
)

charge_identity_max_abs_difference = float(
    np.max(
        np.abs(
            q_plane[finite_q]
            - q_surf_value[finite_q]
        )
    )
)

print(
    "Pearson(charge_raw, SURFMAP value):",
    charge_value_pearson,
)

print(
    "Best linear relation:"
)

print(
    "SURFMAP value ≈",
    charge_slope,
    "* charge_raw +",
    charge_intercept,
)

print(
    "Maximum |charge_raw - SURFMAP value|:",
    charge_identity_max_abs_difference,
)


Pearson(charge_raw, SURFMAP value): 0.9999999999999999
Best linear relation:
SURFMAP value ≈ 0.9999999999999972 * charge_raw + 8.655628631540066e-18
Maximum |charge_raw - SURFMAP value|: 0.0



# PART VIII — Corrected spatial charge-pattern comparison

## 17. Why the previous direct interpolation is replaced

When several physical points share the same SURFMAP 2D coordinate, directly passing all duplicate coordinates to a scattered-data interpolator can make the result dependent on how duplicate positions are handled internally.

That is not ideal for a scientific comparison.

### New strategy

Both maps are processed in exactly the same way:

1. define one regular physical grid in Å;
2. assign every point to a grid cell;
3. if several points fall in the same cell, calculate their **mean charge** explicitly;
4. apply the same Gaussian smoothing to both maps;
5. divide smoothed charge by smoothed occupancy, so empty space does not artificially dilute the signal;
6. compare only locations that are supported in both maps.

### Why several smoothing scales?

A charge patch relevant to SURFMATCH has a physical size.

Fine-scale agreement at 0.5 Å may be sensitive to map discretization, while agreement at 5 Å asks whether larger electrostatic domains are still spatially recognizable.

The multiscale comparison therefore tells us **at what physical resolution SURFMAP begins to preserve the charge landscape**.


In [19]:

q = matched[
    "charge_raw"
].to_numpy(dtype=float)

finite_charge = np.isfinite(q)

P_q = P[finite_charge]
S_q = S_aligned[finite_charge]
q = q[finite_charge]

q_absmax = float(
    np.max(
        np.abs(q)
    )
)

if q_absmax == 0:

    q_norm = np.zeros_like(q)

else:

    q_norm = (
        q / q_absmax
    )


xmin = float(
    P_q[:, 0].min()
)

xmax = float(
    P_q[:, 0].max()
)

ymin = float(
    P_q[:, 1].min()
)

ymax = float(
    P_q[:, 1].max()
)


grid_x = np.arange(
    xmin,
    xmax + CHARGE_GRID_STEP_A,
    CHARGE_GRID_STEP_A,
)

grid_y = np.arange(
    ymin,
    ymax + CHARGE_GRID_STEP_A,
    CHARGE_GRID_STEP_A,
)


def rasterize_points(
    points,
    values,
    grid_x,
    grid_y,
):
    """
    Explicitly aggregate duplicate/colliding points.

    Returns:
        value_sum per cell
        point_count per cell
    """

    step_x = (
        grid_x[1] - grid_x[0]
    )

    step_y = (
        grid_y[1] - grid_y[0]
    )

    sums = np.zeros(
        (
            len(grid_y),
            len(grid_x),
        ),
        dtype=float,
    )

    counts = np.zeros_like(
        sums
    )

    ix = np.floor(
        (
            points[:, 0]
            - grid_x[0]
        )
        / step_x
    ).astype(int)

    iy = np.floor(
        (
            points[:, 1]
            - grid_y[0]
        )
        / step_y
    ).astype(int)

    valid = (
        (ix >= 0)
        & (ix < len(grid_x))
        & (iy >= 0)
        & (iy < len(grid_y))
        & np.isfinite(values)
    )

    np.add.at(
        sums,
        (
            iy[valid],
            ix[valid],
        ),
        values[valid],
    )

    np.add.at(
        counts,
        (
            iy[valid],
            ix[valid],
        ),
        1.0,
    )

    return sums, counts


plane_charge_sum, plane_count = (
    rasterize_points(
        P_q,
        q_norm,
        grid_x,
        grid_y,
    )
)

surf_charge_sum, surf_count = (
    rasterize_points(
        S_q,
        q_norm,
        grid_x,
        grid_y,
    )
)


def normalized_gaussian_field(
    charge_sum,
    counts,
    sigma_A,
    grid_step_A,
):
    sigma_pixels = (
        sigma_A / grid_step_A
    )

    numerator = gaussian_filter(
        charge_sum,
        sigma=sigma_pixels,
        mode="constant",
        cval=0.0,
    )

    support = gaussian_filter(
        counts,
        sigma=sigma_pixels,
        mode="constant",
        cval=0.0,
    )

    field = np.full_like(
        numerator,
        np.nan,
    )

    valid = (
        support
        > SMOOTHED_SUPPORT_THRESHOLD
    )

    field[valid] = (
        numerator[valid]
        / support[valid]
    )

    return field, support


charge_scale_rows = []
charge_fields = {}

for smoothing_A in CHARGE_SMOOTHING_SCALES_A:

    plane_field, plane_support = (
        normalized_gaussian_field(
            plane_charge_sum,
            plane_count,
            sigma_A=smoothing_A,
            grid_step_A=CHARGE_GRID_STEP_A,
        )
    )

    surf_field, surf_support = (
        normalized_gaussian_field(
            surf_charge_sum,
            surf_count,
            sigma_A=smoothing_A,
            grid_step_A=CHARGE_GRID_STEP_A,
        )
    )

    common_mask = (
        np.isfinite(
            plane_field
        )
        &
        np.isfinite(
            surf_field
        )
        &
        (
            plane_support
            > SMOOTHED_SUPPORT_THRESHOLD
        )
        &
        (
            surf_support
            > SMOOTHED_SUPPORT_THRESHOLD
        )
    )

    n_common = int(
        common_mask.sum()
    )

    if n_common < 3:
        continue

    qp = plane_field[
        common_mask
    ]

    qs = surf_field[
        common_mask
    ]

    pearson_r = float(
        pearsonr(
            qp,
            qs,
        ).statistic
    )

    spearman_rho = float(
        spearmanr(
            qp,
            qs,
        ).statistic
    )

    difference = (
        qs - qp
    )

    rmse = float(
        np.sqrt(
            np.mean(
                difference ** 2
            )
        )
    )

    mae = float(
        np.mean(
            np.abs(
                difference
            )
        )
    )

    sign_mask = (
        (
            np.abs(qp)
            >= SIGN_THRESHOLD_NORMALIZED
        )
        |
        (
            np.abs(qs)
            >= SIGN_THRESHOLD_NORMALIZED
        )
    )

    if sign_mask.sum() > 0:

        sign_agreement = float(
            100
            * np.mean(
                np.sign(
                    qp[sign_mask]
                )
                ==
                np.sign(
                    qs[sign_mask]
                )
            )
        )

    else:

        sign_agreement = np.nan

    charge_scale_rows.append({
        "smoothing_scale_A":
            float(smoothing_A),

        "common_grid_cells":
            n_common,

        "pearson_r":
            pearson_r,

        "spearman_rho":
            spearman_rho,

        "rmse_normalized_charge":
            rmse,

        "mae_normalized_charge":
            mae,

        "sign_agreement_pct":
            sign_agreement,
    })

    charge_fields[
        float(smoothing_A)
    ] = {
        "plane":
            plane_field,

        "surfmap":
            surf_field,

        "common_mask":
            common_mask,
    }


charge_multiscale_df = (
    pd.DataFrame(
        charge_scale_rows
    )
)

display(
    charge_multiscale_df
)


,smoothing_scale_A,common_grid_cells,pearson_r,spearman_rho,rmse_normalized_charge,mae_normalized_charge,sign_agreement_pct
0,0.5,5602,0.074906,0.125541,0.274221,0.180472,60.665236
1,1.0,6322,0.094490,0.133121,0.165474,0.116617,58.091371
2,2.0,7739,0.090856,0.182199,0.082520,0.056414,62.303797
3,3.0,9070,0.123102,0.252526,0.055937,0.037117,66.742988
4,5.0,11643,0.408240,0.473344,0.031034,0.019597,99.448656



## 18. Charge similarity as a function of physical resolution

If SURFMAP preserves only broad electrostatic domains but loses fine local structure, we expect:

- low correlation at small smoothing scales;
- progressively better correlation as the physical smoothing scale increases.

If correlation remains poor even at several Å, then the mapping has altered electrostatic organization at a scale likely to matter for surface matching.


In [20]:

fig_charge_scale = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Correlation vs physical scale",
        "Charge error vs physical scale",
    ),
)

fig_charge_scale.add_trace(
    go.Scatter(
        x=charge_multiscale_df[
            "smoothing_scale_A"
        ],
        y=charge_multiscale_df[
            "pearson_r"
        ],
        mode="lines+markers",
        name="Pearson r",
    ),
    row=1,
    col=1,
)

fig_charge_scale.add_trace(
    go.Scatter(
        x=charge_multiscale_df[
            "smoothing_scale_A"
        ],
        y=charge_multiscale_df[
            "spearman_rho"
        ],
        mode="lines+markers",
        name="Spearman ρ",
    ),
    row=1,
    col=1,
)

fig_charge_scale.add_trace(
    go.Scatter(
        x=charge_multiscale_df[
            "smoothing_scale_A"
        ],
        y=charge_multiscale_df[
            "rmse_normalized_charge"
        ],
        mode="lines+markers",
        name="RMSE",
    ),
    row=1,
    col=2,
)

fig_charge_scale.add_trace(
    go.Scatter(
        x=charge_multiscale_df[
            "smoothing_scale_A"
        ],
        y=charge_multiscale_df[
            "mae_normalized_charge"
        ],
        mode="lines+markers",
        name="MAE",
    ),
    row=1,
    col=2,
)

fig_charge_scale.update_xaxes(
    title_text="Gaussian spatial scale (Å)",
    row=1,
    col=1,
)

fig_charge_scale.update_xaxes(
    title_text="Gaussian spatial scale (Å)",
    row=1,
    col=2,
)

fig_charge_scale.update_yaxes(
    title_text="Spatial correlation",
    row=1,
    col=1,
)

fig_charge_scale.update_yaxes(
    title_text="Normalized charge error",
    row=1,
    col=2,
)

fig_charge_scale.update_layout(
    title=(
        "Preservation of electrostatic "
        "patterns at different physical scales"
    ),
    template="plotly_white",
    width=1200,
    height=550,
)

fig_charge_scale.show()



## 19. Side-by-side charge fields at one selected physical scale

This is the most useful visual charge comparison for colleagues.

Both panels:

- use the same physical axes in Å;
- use the same charge normalization;
- use the same raster grid;
- use the same Gaussian physical smoothing scale.

The only difference is the spatial position assigned to the same charged shell points.

Therefore, differences in patch shape or position are directly attributable to the effective SURFMAP representation.


In [21]:

if (
    float(PRIMARY_CHARGE_SCALE_A)
    not in charge_fields
):

    raise ValueError(
        "PRIMARY_CHARGE_SCALE_A must "
        "be included in "
        "CHARGE_SMOOTHING_SCALES_A."
    )


selected_fields = charge_fields[
    float(
        PRIMARY_CHARGE_SCALE_A
    )
]

Q_plane = selected_fields[
    "plane"
]

Q_surf = selected_fields[
    "surfmap"
]

common_charge_mask = (
    selected_fields[
        "common_mask"
    ]
)

charge_scale = [
    [0.00, "rgb(230,0,0)"],
    [0.25, "rgb(255,160,160)"],
    [0.50, "rgb(255,255,255)"],
    [0.75, "rgb(160,160,255)"],
    [1.00, "rgb(0,0,230)"],
]


fig_charge_fields = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "NYLON_PLANE",
        "SURFMAP aligned",
    ),
)

fig_charge_fields.add_trace(
    go.Heatmap(
        x=grid_x,
        y=grid_y,
        z=Q_plane,
        zmin=-1,
        zmax=1,
        colorscale=charge_scale,
        showscale=False,
        hovertemplate=(
            "u = %{x:.2f} Å<br>"
            "v = %{y:.2f} Å<br>"
            "charge field = %{z:.4f}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig_charge_fields.add_trace(
    go.Heatmap(
        x=grid_x,
        y=grid_y,
        z=Q_surf,
        zmin=-1,
        zmax=1,
        colorscale=charge_scale,
        colorbar=dict(
            title="Normalized charge"
        ),
        hovertemplate=(
            "u = %{x:.2f} Å<br>"
            "v = %{y:.2f} Å<br>"
            "charge field = %{z:.4f}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig_charge_fields.update_xaxes(
    scaleanchor="y",
    scaleratio=1,
    row=1,
    col=1,
)

fig_charge_fields.update_xaxes(
    scaleanchor="y2",
    scaleratio=1,
    row=1,
    col=2,
)

fig_charge_fields.update_layout(
    title=(
        "Charge pattern comparison at "
        f"{PRIMARY_CHARGE_SCALE_A:g} Å "
        "physical smoothing"
    ),
    template="plotly_white",
    width=1250,
    height=650,
)

fig_charge_fields.show()



## 20. Charge-field difference map

For the selected physical smoothing scale:

\[
\Delta Q(u,v)
=
Q_{SURFMAP}(u,v)
-
Q_{PLANE}(u,v)
\]

White/near-zero regions indicate good local preservation.

Large positive or negative differences identify locations where the effective SURFMAP representation places a different electrostatic environment than the planar geometry.


In [22]:

Q_difference = (
    Q_surf - Q_plane
)

Q_difference[
    ~common_charge_mask
] = np.nan

finite_difference = (
    Q_difference[
        np.isfinite(
            Q_difference
        )
    ]
)

difference_limit = float(
    np.max(
        np.abs(
            finite_difference
        )
    )
)

fig_charge_difference = go.Figure(
    go.Heatmap(
        x=grid_x,
        y=grid_y,
        z=Q_difference,
        zmin=-difference_limit,
        zmax=difference_limit,
        colorscale="RdBu",
        colorbar=dict(
            title=(
                "SURFMAP - PLANE"
            )
        ),
        hovertemplate=(
            "u = %{x:.2f} Å<br>"
            "v = %{y:.2f} Å<br>"
            "ΔQ = %{z:.4f}"
            "<extra></extra>"
        ),
    )
)

fig_charge_difference.update_layout(
    title=(
        "Spatial charge-field difference "
        f"at {PRIMARY_CHARGE_SCALE_A:g} Å scale"
    ),
    xaxis=dict(
        title="u (Å)",
        scaleanchor="y",
        scaleratio=1,
    ),
    yaxis_title="v (Å)",
    template="plotly_white",
    width=900,
    height=900,
)

fig_charge_difference.show()



# PART IX — Point-level charge displacement

## 21. Charge-weighted geometric displacement

The field comparison above evaluates spatial patterns.

A complementary quantity is the physical displacement of strongly charged points.

For each point we already know:

\[
e_i=
\|S_{i,aligned}-P_i\|
\]

We calculate a weighted RMS using:

\[
w_i=|q_i|
\]

so that points with very small charge contribute less than strongly charged points.

This does not measure field correlation; it answers:

> **How far are electrostatically important shell points moved by the SURFMAP representation?**


In [23]:

q_weight = np.abs(
    q_plane
)

if q_weight.sum() > 0:

    charge_weighted_rms_A = float(
        np.sqrt(
            np.sum(
                q_weight
                * residual_A ** 2
            )
            /
            np.sum(q_weight)
        )
    )

else:

    charge_weighted_rms_A = np.nan


def weighted_centroid(
    points,
    weights,
):

    weights = np.asarray(
        weights,
        dtype=float,
    )

    if weights.sum() <= 0:

        return np.array([
            np.nan,
            np.nan,
        ])

    return (
        np.sum(
            points
            * weights[:, None],
            axis=0,
        )
        / weights.sum()
    )


q_normalized_points = (
    q_plane
    / np.max(
        np.abs(q_plane)
    )
)

positive_weight = np.clip(
    q_normalized_points,
    0,
    None,
)

negative_weight = np.clip(
    -q_normalized_points,
    0,
    None,
)

positive_center_plane = (
    weighted_centroid(
        P,
        positive_weight,
    )
)

positive_center_surf = (
    weighted_centroid(
        S_aligned,
        positive_weight,
    )
)

negative_center_plane = (
    weighted_centroid(
        P,
        negative_weight,
    )
)

negative_center_surf = (
    weighted_centroid(
        S_aligned,
        negative_weight,
    )
)

positive_centroid_shift_A = float(
    np.linalg.norm(
        positive_center_surf
        - positive_center_plane
    )
)

negative_centroid_shift_A = float(
    np.linalg.norm(
        negative_center_surf
        - negative_center_plane
    )
)

print(
    "Charge-weighted RMS displacement:",
    charge_weighted_rms_A,
    "Å",
)

print(
    "Positive-charge centroid shift:",
    positive_centroid_shift_A,
    "Å",
)

print(
    "Negative-charge centroid shift:",
    negative_centroid_shift_A,
    "Å",
)


Charge-weighted RMS displacement: 15.242671740639944 Å
Positive-charge centroid shift: 1.3776927409633006 Å
Negative-charge centroid shift: 0.9926236993031919 Å



# PART X — Quantitative summary for colleagues

## 22. How to interpret the groups of metrics

### A. Data integrity

These must be correct before any scientific conclusion is made:

- point matching coverage;
- XYZ matching error;
- charge-value consistency.

### B. Global shape distortion

Use:

- RMS positional error;
- P95 positional error;
- NRMSD;
- geometry overlay.

These are the strongest measures of overall representation mismatch.

### C. Quantization / information loss

Use:

- number of unique SURFMAP positions;
- collision fraction;
- maximum multiplicity;
- effective coordinate step in Å.

These tell us whether the available SURFMAP coordinate representation has already lost point-level spatial resolution.

### D. Central-region fidelity

Use:

- central physical radius;
- best-fit RMS;
- best-fit P95;
- best-fit uniform scale.

This is the correct diagnostic for asking whether a future central ROI could be substantially more accurate than the full map.

### E. Local geometry at physically relevant scales

Use the multiscale pair-distance table.

Fine scales reveal discretization sensitivity.

The 2–5 Å, 5–10 Å and 10–15 Å bands are especially useful for distinguishing simple pixel collapse from larger-scale shape deformation.

### F. Charge pattern preservation

Use the multiscale charge table and the charge-field plots.

If correlation improves only after very large smoothing, SURFMAP preserves broad charge polarity but not the fine spatial pattern.

No automatic pass/fail threshold is imposed because the acceptable error ultimately depends on how sensitive the final SURFMATCH score is to these geometric changes.


In [24]:

geometry_summary = {
    "matched_points":
        int(len(matched)),

    "matching_coverage_pct":
        float(
            100 * coverage
        ),

    "max_xyz_match_error_A":
        float(
            matched[
                "xyz_match_error_A"
            ].max()
        ),

    "alignment_type":
        similarity_fit["label"],

    "alignment_detR":
        float(detR),

    "alignment_rotation_angle_deg":
        (
            float(rotation_angle_deg)
            if np.isfinite(
                rotation_angle_deg
            )
            else None
        ),

    "alignment_uniform_scale":
        float(
            similarity_fit["scale"]
        ),

    "global_RMS_error_A":
        similarity_rms_A,

    "global_P95_error_A":
        similarity_p95_A,

    "global_max_error_A":
        similarity_max_A,

    "global_NRMS_RMSradius_pct":
        float(
            nrms_rms_radius_pct
        ),

    "global_NRMS_bboxdiag_pct":
        float(
            nrms_bbox_diag_pct
        ),

    "procrustes_disparity":
        procrustes_disparity,
}


quantization_summary = {
    "integer_coordinate_fraction_pct":
        float(
            100
            * integer_fraction
        ),

    "unique_SURFMAP_positions":
        n_unique_surf_positions,

    "matched_points":
        n_total_points,

    "unique_position_fraction_pct":
        float(
            100
            * n_unique_surf_positions
            / n_total_points
        ),

    "points_in_shared_positions_pct":
        float(
            100
            * shared_position_fraction
        ),

    "max_position_multiplicity":
        max_position_multiplicity,

    "smallest_raw_coordinate_step":
        float(
            minimum_raw_grid_step
        ),

    "effective_grid_step_A":
        float(
            effective_grid_step_A
        ),
}


charge_summary = {
    "charge_value_pearson":
        charge_value_pearson,

    "charge_value_max_identity_difference":
        charge_identity_max_abs_difference,

    "charge_weighted_RMS_displacement_A":
        float(
            charge_weighted_rms_A
        ),

    "positive_charge_centroid_shift_A":
        positive_centroid_shift_A,

    "negative_charge_centroid_shift_A":
        negative_centroid_shift_A,
}


summary_rows = []

for key, value in geometry_summary.items():

    summary_rows.append({
        "category":
            "global_geometry",

        "metric":
            key,

        "value":
            value,
    })

for key, value in quantization_summary.items():

    summary_rows.append({
        "category":
            "SURFMAP_quantization",

        "metric":
            key,

        "value":
            value,
    })

for key, value in charge_summary.items():

    summary_rows.append({
        "category":
            "charge",

        "metric":
            key,

        "value":
            value,
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(summary_df)


,category,metric,value
0,global_geometry,matched_points,6148
1,global_geometry,matching_coverage_pct,100.0
2,global_geometry,max_xyz_match_error_A,0.0
3,global_geometry,alignment_type,rotation
4,global_geometry,alignment_detR,1.0
5,global_geometry,alignment_rotation_angle_deg,-89.026825
6,global_geometry,alignment_uniform_scale,0.364771
7,global_geometry,global_RMS_error_A,15.286261
8,global_geometry,global_P95_error_A,24.775964
9,global_geometry,global_max_error_A,37.495155



# PART XI — Export all outputs

The exported files are designed so that the results can be shown to colleagues without rerunning every intermediate calculation.

### Recommended figures for a discussion

1. `00_alignment_explained.html`  
   Shows what alignment removes and what shape deformation remains.

2. `01_positional_error_comparison.html`  
   Shows the same positional error on both NYLON_PLANE and the aligned SURFMAP map, so we can identify both which physical regions are distorted and which SURFMAP regions are geometrically unreliable.

3. `02_central_region_best_fit.html`  
   Shows whether progressively larger central physical regions remain accurate.

4. `03_multiscale_distance_distortion.html`  
   Shows local compression/expansion at different physical length scales.

5. `04_charge_similarity_vs_scale.html`  
   Shows whether charge patterns are preserved at fine or coarse spatial resolution.

6. `05_charge_fields.html`  
   Side-by-side spatial charge fields in the same physical coordinate system.

7. `06_charge_field_difference.html`  
   Shows where electrostatic pattern differences occur.

### Recommended numerical tables

- `comparison_summary.csv`
- `central_region_best_fit.csv`
- `multiscale_geometry_distortion.csv`
- `multiscale_charge_similarity.csv`
- `matched_points.csv`


In [25]:

# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

matched_csv = (
    OUTPUT_DIR
    / "matched_points.csv"
)

summary_csv = (
    OUTPUT_DIR
    / "comparison_summary.csv"
)

central_csv = (
    OUTPUT_DIR
    / "central_region_best_fit.csv"
)

multiscale_geom_csv = (
    OUTPUT_DIR
    / "multiscale_geometry_distortion.csv"
)

multiscale_charge_csv = (
    OUTPUT_DIR
    / "multiscale_charge_similarity.csv"
)

summary_json = (
    OUTPUT_DIR
    / "comparison_summary.json"
)


matched.to_csv(
    matched_csv,
    index=False,
)

summary_df.to_csv(
    summary_csv,
    index=False,
)

central_region_df.to_csv(
    central_csv,
    index=False,
)

multiscale_geometry_df.to_csv(
    multiscale_geom_csv,
    index=False,
)

charge_multiscale_df.to_csv(
    multiscale_charge_csv,
    index=False,
)


with summary_json.open("w") as f:

    json.dump(
        {
            "global_geometry":
                geometry_summary,

            "quantization":
                quantization_summary,

            "charge":
                charge_summary,

            "inputs": {
                "plane_points_csv":
                    str(
                        PLANE_POINTS_CSV
                    ),

                "plane_metadata_json":
                    str(
                        PLANE_METADATA_JSON
                    ),

                "surfmap_coord_list":
                    str(
                        coord_list_path
                    ),

                "surfmap_shell_pdb":
                    str(
                        shell_path
                    ),

                "index_offset":
                    int(
                        INDEX_OFFSET
                    ),
            },
        },
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Interactive HTML figures
# ------------------------------------------------------------

alignment_html = (
    OUTPUT_DIR
    / "00_alignment_explained.html"
)

error_html = (
    OUTPUT_DIR
    / "01_positional_error_comparison.html"
)

central_html = (
    OUTPUT_DIR
    / "02_central_region_best_fit.html"
)

central_scale_html = (
    OUTPUT_DIR
    / "02b_central_region_scale.html"
)

multiscale_geom_html = (
    OUTPUT_DIR
    / "03_multiscale_distance_distortion.html"
)

collapse_html = (
    OUTPUT_DIR
    / "03b_pair_collapse_by_scale.html"
)

charge_scale_html = (
    OUTPUT_DIR
    / "04_charge_similarity_vs_scale.html"
)

charge_fields_html = (
    OUTPUT_DIR
    / "05_charge_fields.html"
)

charge_difference_html = (
    OUTPUT_DIR
    / "06_charge_field_difference.html"
)


fig_alignment.write_html(
    alignment_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_error.write_html(
    error_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_central.write_html(
    central_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_central_scale.write_html(
    central_scale_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_multiscale_geom.write_html(
    multiscale_geom_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_collapse.write_html(
    collapse_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_charge_scale.write_html(
    charge_scale_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_charge_fields.write_html(
    charge_fields_html,
    include_plotlyjs="cdn",
    full_html=True,
)

fig_charge_difference.write_html(
    charge_difference_html,
    include_plotlyjs="cdn",
    full_html=True,
)


print("Saved all outputs in:")
print(OUTPUT_DIR)

print()
print("Main files:")

for path in [
    summary_csv,
    central_csv,
    multiscale_geom_csv,
    multiscale_charge_csv,
    alignment_html,
    error_html,
    central_html,
    multiscale_geom_html,
    charge_scale_html,
    charge_fields_html,
    charge_difference_html,
]:

    print(" -", path)


Saved all outputs in:
/home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2

Main files:
 - /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2/comparison_summary.csv
 - /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2/central_region_best_fit.csv
 - /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2/multiscale_geometry_distortion.csv
 - /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2/multiscale_charge_similarity.csv
 - /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_plane_projection/nylon_plane_vs_surfmap_corrected_comparison_2/00_alignment_explained.html
 - /home/chiara/Desktop/COMEN


# Final interpretation guide

When presenting the results, avoid reducing everything immediately to one generic “percentage error”.

Instead, answer the following questions in sequence.

## 1. Are we comparing the correct points?

The matching coverage should be close to 100%, with negligible XYZ mismatch.

If this fails, no geometric conclusion should be made.

## 2. Is the full SURFMAP representation geometrically close to the physical plane?

Use:

- alignment overlay;
- global RMS positional error;
- P95 positional error;
- NRMSD.

These answer the global shape question.

## 3. Could the apparent error simply come from SURFMAP pixel/grid quantization?

Compare:

- effective SURFMAP grid step in Å;
- global positional error;
- coordinate-collision statistics.

If the global error is tens of times larger than one effective grid step, quantization alone cannot explain the global mismatch.

## 4. Is a central region substantially better?

Use `central_region_best_fit.csv` and the corresponding plot.

This is more appropriate than examining a globally aligned radial profile because each central region is allowed its own best harmless similarity alignment.

If a small central region has low absolute RMS/P95 error and error rises as the region expands, a central ROI may be scientifically defensible.

## 5. Is local geometry preserved at the length scales relevant to SURFMATCH?

Use the multiscale pair-distance analysis.

Very fine scales can be dominated by grid collisions.

If large stretch/compression remains at 5–10 Å or 10–15 Å, then the problem is not merely pixelization: larger spatial relationships are being distorted.

## 6. Are electrostatic patterns preserved?

Use charge similarity versus smoothing scale.

A useful interpretation is:

- good correlation already at 1–2 Å  
  → fine electrostatic pattern is largely preserved;

- poor correlation at 1–2 Å but strong correlation at 5 Å  
  → broad charge domains are preserved but fine patch geometry is lost;

- poor correlation even at 5 Å  
  → electrostatic organization is substantially rearranged at a relatively large physical scale.

## 7. Can SURFMAP be used in SURFMATCH?

This notebook quantifies the representation error.

The final software decision should then consider:

- the physical length scale used by the matching score;
- whether matching uses fine local charge arrangements or only broad domains;
- whether the intended analysis uses the complete nylon face or only a central region.

If the representation error is large at the same spatial scale used by SURFMATCH, the planar nylon representation is the safer geometric basis.

If error becomes small within a well-defined central physical region, a restricted SURFMAP representation may still be usable and can be tested separately.
